In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
from pathlib import Path
import pandas as pd
import os
import rasterio as rio
from glob import glob
from scipy.stats import mannwhitneyu
import seaborn as sns
import torch
from torch_geometric.utils import k_hop_subgraph
from torch_sparse import SparseTensor
from tqdm import tqdm
import xarray as xr

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None), model='routed_random')[['Q_pred_m3s', 'q_pred_mmd', 'S_pred_m3', 'is_lake']].drop_vars(
        ['x', 'y', 'model', 'hru_i'])

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

hrus_lakes = hrus.loc[ds.hru.where(ds.is_lake.load(), drop=True)]

rivers = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)[['CUM_LEN', 'geometry']]

hrus_outline = hrus.geometry.unary_union

S_lakes = ds.S_pred_m3.sel(hru=hrus_lakes.index)
S_q = S_lakes.load().quantile([0.02, 0.98], 'time').compute()
S_lakes = np.clip((S_lakes - S_q.isel(quantile=0)) / (S_q.isel(quantile=1) - S_q.isel(quantile=0)), 0, 1)

# rivers_cut = gpd.overlay(
#     rivers,
#     hrus.reset_index()[['EZGNR', 'geometry']],
#     how='intersection'
# ).set_index('EZGNR', drop=False)

# common_hrus_poly = hrus.index.intersection(ds.hru.values)
# common_hrus_rivers = rivers_cut.index.intersection(ds.hru.values)

# rivers_cut = rivers_cut.loc[common_hrus_rivers]

# rivers_cut = rivers_cut.explode(index_parts=False).reset_index(drop=True)



In [ ]:
q_cmap = 'inferno'
q_norm = mpl.colors.LogNorm(vmin=1, vmax=50)
q_alpha = 0.5

Q_cmap = 'YlGnBu_r'
Q_norm = mpl.colors.LogNorm(vmin=0.1, vmax=3000)
Q_alpha = 1.0

S_cmap = 'BrBG'
S_norm = mpl.colors.Normalize(vmin=0, vmax=1)
S_alpha = 0.5

times = ds.time.sel(time='2007')

for i, time in tqdm(enumerate(times), total=len(times), ncols=120):

    dss = ds.sel(time=time)

    hrus_q = enrich(dss.q_pred_mmd, hrus, 'q')
    hrus_Q = enrich(dss.Q_pred_m3s, hrus, 'Q')
    hrus_S = enrich(S_lakes.sel(time=time), hrus_lakes, 'S')

    rivers_cut = gpd.overlay(
        rivers,
        hrus_Q.reset_index()[['EZGNR', 'Q', 'geometry']],
        how='intersection'
    )
    rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)
    rivers_Q = rivers_Q.sort_values('Q', ascending=True)

    rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

    fig, ax = plt.subplots(figsize=(7, 4), subplot_kw={'projection': lv95, 'frameon': True})

    ax.set_global()

    hrus_q.plot(
        column='q',
        ec='k',
        lw=0.05,
        cmap=q_cmap,
        alpha=q_alpha,
        antialiased=True,
        norm=q_norm,
        ax=ax
    )

    q_axins = ax.inset_axes((0.04, 0.75, 0.16, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=q_norm, cmap=q_cmap)
    q_cbar = plt.colorbar(
        mappable,
        cax=q_axins,
        alpha=q_alpha,
        orientation='horizontal',   # or 'vertical'
        extend='max',
        label='Runoff (mm d$^{-1}$)'
    )

    rivers_Q.plot(
        color='k',
        linewidth=rw + 0.5,
        zorder=2,# 
        capstyle='round',
        legend=False,
        ax=ax)

    rivers_Q.plot(
        column='Q',
        linewidth=rw + 0.1,
        norm=Q_norm,
        zorder=2,
        capstyle='round',
        fc='none',
        cmap=Q_cmap,
        legend=False,
        ax=ax)

    Q_axins = ax.inset_axes((0.04, 0.89, 0.16, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=Q_norm, cmap=Q_cmap)
    Q_cbar = plt.colorbar(
        mappable,
        cax=Q_axins,
        orientation='horizontal',   # or 'vertical'
        extend='max',
        label='Discharge (m$^3$ s$^{-1}$)'
    )

    hrus_S.plot(
        ec='none',
        fc='white',
        lw=0.0,
        ax=ax,
        zorder=30
    )
    hrus_S.plot(
        column='S',
        ec='none',
        lw=0.0,
        ax=ax,
        norm=S_norm,
        cmap=S_cmap,
        alpha=S_alpha,
        zorder=30
    )
    hrus_S.plot(
        ec='k',
        fc='none',
        lw=0.2,
        ax=ax,
        zorder=30
    )

    S_axins = ax.inset_axes((0.77, 0.16, 0.16, 0.02), zorder=30)
    mappable = mpl.cm.ScalarMappable(norm=S_norm, cmap=S_cmap)
    S_cbar = plt.colorbar(
        mappable,
        cax=S_axins,
        alpha=S_alpha,
        orientation='horizontal',   # or 'vertical'
        extend='neither',
        label='Relative lake vol. (-)'
    )

    ax.plot(*hrus_outline.exterior.xy, color='k', lw=0.5, zorder=35)

    time_string = dss.time.dt.strftime('%Y-%m-%d').item()
    ax.text(0.01, 0.99, f'Date: {time_string}', fontsize=8, transform=ax.transAxes,
        verticalalignment='top', horizontalalignment='left',
        zorder=40
    )

    for cbar in [q_cbar, Q_cbar, S_cbar]:
        cbar.ax.xaxis.label.set_size(7)
        cbar.ax.tick_params(labelsize=7)
        cbar.ax.xaxis.labelpad = 0

    fig.savefig(f'./figures/anim/{i:04d}.png', dpi=300, bbox_inches='tight')

    plt.close()

s = (
    "ffmpeg -y -framerate 6 -pattern_type glob -i './figures/anim/*.png' -c:v libx264 -pix_fmt yuv420p "
    "'./figures/anim/animation.mp4'"
)
os.system(s)